In [1]:
import pandas as pd
import textmining.resources as res

In [2]:
df = pd.read_csv(res.HMDD_DATA, sep='\t')

In [3]:
df.head()

,code,PMID,miRNA,disease,description
0,other,36948853,hsa-mir-40,Liver Cirrhosis,MiR-340 mediates the involvement of high mobil...
1,target gene,36948479,hsa-mir-29b,Zika Virus Infection,Zika virus NS1 suppresses VE-cadherin via hsa-...
2,exosome,36948039,hsa-mir-130b,Wound Healing,ADSCs-derived exosomal H19 accelerates cutaneo...
3,target gene,36947996,hsa-mir-221,Stomach Neoplasms,Downregulation of miR-221-3p promotes the ferr...
4,other,36947456,hsa-mir-497,Pancreatic Neoplasms,Long noncoding RNA CASC9 promotes pancreatic c...


In [4]:
from textmining.config import MirbaseResources
from textmining.normalization_resources import MirResourceLoader
import json

In [5]:
mirbase = MirbaseResources()
norm_resources = MirResourceLoader.load(
        mirna_taxons_path=mirbase.mirna_taxons_path,
        mirna_2_prefix_path=mirbase.mirna_to_prefix_path,
        family_normalizer_path=mirbase.family_norm_path,
        precursor_normalizer_path=mirbase.precursor_norm_path,
        mature_normalizer_path=mirbase.mature_norm_path,
        precursor_ambiguous_path=mirbase.precursor_ambi_path,
        mature_ambiguous_path=mirbase.mature_ambi_path,
        parent_to_child_path=mirbase.parent_to_child_path,
    )


In [6]:
import csv
from collections import Counter, defaultdict

# miRBase dict keys are all lowercase, but miRBase writes viral miRNAs in caps
# (ebv-mir-BART1, kshv-mir-K12-1), so lookup has to be case-insensitive.
# Nothing else is folded: HMDD's hyphen loss (hsa-mir-161 for hsa-mir-16-1) is
# reported as unresolved rather than repaired, since hsa-mir-71 is equally
# readable as a corrupted hsa-mir-7-1 or as cel-mir-71 under a wrong prefix.
def fold(name):
    return name.strip().lower()

def fold_index(keys):
    index = defaultdict(set)
    for key in keys:
        index[fold(key)].add(key)
    return index

precursor_folded = fold_index(norm_resources.precursor_normalizer)
mature_folded = fold_index(norm_resources.mature_normalizer)
family_folded = fold_index(norm_resources.family_normalizer)

print('case-folding collisions:',
      sum(len(names) > 1 for names in precursor_folded.values()),
      sum(len(names) > 1 for names in mature_folded.values()),
      sum(len(names) > 1 for names in family_folded.values()))

case-folding collisions: 0 0 0


In [7]:
body_to_prefixes = defaultdict(set)
accession_to_prefix = {}
with open(mirbase.precursor_path) as f:
    for row in csv.DictReader(f, delimiter='\t'):
        prefix, _, body = row['id'].partition('-')
        accession_to_prefix[row['accession']] = prefix
        if body:
            body_to_prefixes[fold(body)].add(prefix)

family_prefixes = defaultdict(set)
with open(mirbase.parent_to_child_path) as f:
    for row in csv.DictReader(f, delimiter='\t'):
        prefix = accession_to_prefix.get(row['child'])
        if prefix:
            family_prefixes[row['parent']].add(prefix)

print('families with members:', len(family_prefixes))

families with members: 1976


In [8]:
def classify(raw):
    name = raw.strip()
    if name in norm_resources.ambiguous_precursors or name in norm_resources.ambiguous_mature:
        return 'ambiguous', name

    for level, index in (('precursor', norm_resources.precursor_normalizer),
                         ('mature', norm_resources.mature_normalizer)):
        if name in index:
            return f'known: {level}', name

    folded = fold(name)
    for level, index in (('precursor', precursor_folded), ('mature', mature_folded)):
        if folded in index:
            return f'known: {level} (case-folded)', '|'.join(sorted(index[folded]))

    # miRBase names families without a species prefix (mir-15, let-7), so the
    # prefix has to be stripped before the family lookup can hit at all.
    # Requiring a member of the claimed species is a plausibility guard, not an
    # identification: hsa-mir-14 matches family mir-14, but that family is
    # insect-only, so it falls through to the species check rather than being
    # accepted as a human family reference.
    # The guard tests miRBase's family naming, which does not track member
    # naming, so it rejects some legitimate references: hsa-mir-200 fails
    # because the human miR-200 genes sit in family mir-8, while family mir-200
    # holds two sea squirt genes.
    prefix, _, body = name.partition('-')
    families = family_folded.get(fold(body), set())
    for family in families:
        members = family_prefixes.get(norm_resources.family_normalizer[family], set())
        if prefix in members:
            return 'family: has member of this species', family

    others = body_to_prefixes.get(fold(body), set()) - {prefix}
    if others:
        return 'wrong species prefix', ','.join(sorted(others))

    if families:
        return 'family: no member of this species', '|'.join(sorted(families))
    return 'unresolved', name

In [9]:
names = sorted({str(name).strip() for name in df['miRNA']})
verdicts = {name: classify(name) for name in names}

print(f'distinct names: {len(names)}')
for status, n in Counter(status for status, _ in verdicts.values()).most_common():
    print(f'{n:6}  {100 * n / len(names):5.1f}%  {status}')

distinct names: 1895
  1265   66.8%  known: precursor
   295   15.6%  wrong species prefix
   161    8.5%  unresolved
    73    3.9%  known: mature
    55    2.9%  known: precursor (case-folded)
    29    1.5%  family: has member of this species
    14    0.7%  ambiguous
     3    0.2%  family: no member of this species


In [10]:
df['status'] = df['miRNA'].map(lambda name: verdicts[str(name).strip()][0])

print(f'associations: {len(df)}')
for status, n in df['status'].value_counts().items():
    print(f'{n:6}  {100 * n / len(df):5.1f}%  {status}')

associations: 53553
 42843   80.0%  known: precursor
  7533   14.1%  known: mature
  1110    2.1%  family: has member of this species
   923    1.7%  wrong species prefix
   723    1.4%  ambiguous
   275    0.5%  unresolved
   141    0.3%  known: precursor (case-folded)
     5    0.0%  family: no member of this species


In [11]:
residue = Counter()
for name, (status, _) in verdicts.items():
    if status == 'unresolved':
        residue[name] = int((df['miRNA'].str.strip() == name).sum())

print(f'unresolved: {len(residue)} names, {sum(residue.values())} associations')
for name, n in residue.most_common():
    print(f'{n:5}  {name}')

unresolved: 161 names, 275 associations
   12  hsa-mir-516a
   12  hsa-mir-520
    9  hsa-mir-519
    7  hsa-mir-156
    6  hsa-mir-159
    6  hsa-mir-160
    6  hsa-mir-408
    6  hsa-mir-517
    5  hsa-mir-169
    4  hsa-mir-162
    4  hsa-mir-518
    3  14q32 cluster
    3  hcmv-mir-US4-1
    3  hsa-mir-164
    3  hsa-mir-165
    3  hsa-mir-166
    3  hsa-mir-1885
    3  hsa-mir-2911
    3  hsa-mir-397
    3  hsa-mir-4331
    3  hsa-mir-466f
    3  hsa-mir-467a
    3  hsa-mir-516
    3  hsa-mir-5484
    2  C19MC cluster
    2  hsa-let-7m
    2  hsa-mir-134a
    2  hsa-mir-160a
    2  hsa-mir-167
    2  hsa-mir-168
    2  hsa-mir-171f
    2  hsa-mir-172
    2  hsa-mir-172b
    2  hsa-mir-183c
    2  hsa-mir-185a
    2  hsa-mir-23-2
    2  hsa-mir-27d-1
    2  hsa-mir-291
    2  hsa-mir-319
    2  hsa-mir-390
    2  hsa-mir-398a
    2  hsa-mir-44352
    2  hsa-mir-473
    2  hsa-mir-6026
    2  hsa-mir-6511b
    2  hsa-mir-7e
    1  ebv-mir-BART-22
    1  ebv-mir-BHRF1
    1  ebv-mir-

In [12]:
diseases = df['disease']
diseases.head(30)

0                    Liver Cirrhosis
1               Zika Virus Infection
2                      Wound Healing
3                  Stomach Neoplasms
4               Pancreatic Neoplasms
5     Carcinoma, Non-Small-Cell Lung
6                    Liver Neoplasms
7                      Neuroblastoma
8               Spinal Cord Injuries
9         Uterine Cervical Neoplasms
10           Hypertension, Pulmonary
11           Hypertension, Pulmonary
12                 Stomach Neoplasms
13                    Retinoblastoma
14              Hepatitis B, Chronic
15              Hepatitis B, Chronic
16              Hepatitis B, Chronic
17              Hepatitis B, Chronic
18              Hepatitis B, Chronic
19              Hepatitis B, Chronic
20              Hepatitis B, Chronic
21              Hepatitis B, Chronic
22              Hepatitis B, Chronic
23              Hepatitis B, Chronic
24              Hepatitis B, Chronic
25              Hepatitis B, Chronic
26              Hepatitis B, Chronic
2

In [13]:
len(set(diseases))

2379

In [14]:
import re

def normalize_disease(raw: str):
    """Disease lookup key: case, hyphen/slash and whitespace insensitive.

    Unlike miRNA names, hyphens in disease terms are orthographic - MeSH writes
    "non-small-cell lung" where MONDO writes "non-small cell lung" - so folding
    them is safe here in a way it was not for miRNA copy-number suffixes.
    """
    return re.sub(r'\s+', ' ', re.sub(r'[-/]', ' ', raw.strip().lower())).strip()

In [15]:
syn_to_ids = defaultdict(set)
with open(res.DISEASE_SYNS, 'r') as d:
    for line in d:
        term_id, syn_line = line.strip().split(':', 1)
        for syn in syn_line.split('|'):
            syn_to_ids[normalize_disease(syn)].add(term_id)

print(f'synonym strings: {len(syn_to_ids)}   '
      f'mapping to >1 MONDO id: {sum(len(ids) > 1 for ids in syn_to_ids.values())}')

synonym strings: 75209   mapping to >1 MONDO id: 24


In [16]:
def drop_plural(raw: str):
    if raw.endswith('s') and not raw.endswith('ss'):
        return raw[:-1]
    return raw

In [17]:
def deinvert_comma(raw: str):
    parts = [p.strip() for p in raw.split(',')]
    return ' '.join(reversed(parts)) if len(parts) > 1 else raw

In [18]:
def match_disease(raw: str):
    """Map an HMDD disease string to MONDO ids, recording which rule matched.

    Rules are tried most to least literal, so the reported rule is the weakest
    transformation needed. A string matching several MONDO terms is reported as
    ambiguous rather than resolved to an arbitrary one.
    """
    name = normalize_disease(raw)
    for rule, candidate in (('exact', name),
                            ('plural', drop_plural(name)),
                            ('deinverted', deinvert_comma(name)),
                            ('deinverted+plural', drop_plural(deinvert_comma(name)))):
        ids = syn_to_ids.get(candidate)
        if ids:
            return ('ambiguous' if len(ids) > 1 else rule), frozenset(ids)
    return 'unmatched', frozenset()

disease_verdicts = {d: match_disease(d) for d in set(diseases)}
df['disease_rule'] = df['disease'].map(lambda d: disease_verdicts[d][0])
df['mondo_ids'] = df['disease'].map(lambda d: disease_verdicts[d][1])

print(f'distinct diseases: {len(disease_verdicts)}')
for rule, n in Counter(r for r, _ in disease_verdicts.values()).most_common():
    print(f'{n:6}  {100 * n / len(disease_verdicts):5.1f}%  {rule}')

distinct diseases: 2379
  1043   43.8%  unmatched
   924   38.8%  exact
   244   10.3%  deinverted
   150    6.3%  plural
    15    0.6%  deinverted+plural
     3    0.1%  ambiguous


In [19]:
print(f'associations: {len(df)}')
for rule, n in df['disease_rule'].value_counts().items():
    print(f'{n:6}  {100 * n / len(df):5.1f}%  {rule}')

mapped = df[df['mondo_ids'].map(len) == 1]
print(f'\nmapped to exactly one MONDO id: {len(mapped)} associations, '
      f'{len(set().union(*mapped["mondo_ids"]))} distinct terms')
for disease, ids in list(dict(zip(mapped['disease'], mapped['mondo_ids'])).items())[:8]:
    print(f'  {disease:42} {next(iter(ids))}')

associations: 53553
 22381   41.8%  exact
 12287   22.9%  unmatched
 10158   19.0%  plural
  8654   16.2%  deinverted
    62    0.1%  deinverted+plural
    11    0.0%  ambiguous

mapped to exactly one MONDO id: 41255 associations, 1287 distinct terms
  Zika Virus Infection                       MONDO_0018661
  Stomach Neoplasms                          MONDO_0021085
  Pancreatic Neoplasms                       MONDO_0021040
  Carcinoma, Non-Small-Cell Lung             MONDO_0005233
  Liver Neoplasms                            MONDO_0024477
  Neuroblastoma                              MONDO_0005072
  Hypertension, Pulmonary                    MONDO_0005149
  Retinoblastoma                             MONDO_0008380


In [20]:
unmatched = Counter()
for disease, (rule, _) in disease_verdicts.items():
    if rule == 'unmatched':
        unmatched[normalize_disease(disease)] += int((df['disease'] == disease).sum())

print(f'unmatched: {len(unmatched)} diseases, {sum(unmatched.values())} associations')
for disease, n in unmatched.most_common(20):
    print(f'{n:5}  {disease}')

unmatched: 1029 diseases, 12287 associations
  971  prostatic neoplasms
  902  uterine cervical neoplasms
  543  pancreatic carcinoma
  429  diabetic nephropathies
  425  triple negative breast neoplasms
  414  inflammation
  378  sepsis
  236  spinal cord injuries
  206  liver cirrhosis
  185  cardiomegaly
  182  myocardial reperfusion injury
  136  fibrosis
  134  acquired immunodeficiency syndrome
  133  carcinoma, ovarian epithelial
  122  hepatitis c
  118  hypertension
  118  precursor cell lymphoblastic leukemia lymphoma
  117  psoriasis 1
  102  reperfusion injury
   97  reperfusion injury, cerebral ischemia


In [21]:
# Both halves use the same standard: a row counts as detected only if the
# entity resolves to a single identifier. Ambiguous miRNA names (conflicting
# accessions across releases), non-human miRNAs and diseases matching several
# MONDO terms are all excluded, since none of them yield one node to attach
# the association to.
MIRNA_DETECTED = {'known: precursor', 'known: mature',
                  'known: precursor (case-folded)', 'known: mature (case-folded)',
                  'family: has member of this species'}

mirna_ok = df['status'].isin(MIRNA_DETECTED)
disease_ok = ~df['disease_rule'].isin(['unmatched', 'ambiguous'])

print(f'associations: {len(df)}')
for label, mask in (('miRNA resolved', mirna_ok),
                    ('disease resolved', disease_ok),
                    ('both (usable association)', mirna_ok & disease_ok),
                    ('miRNA only', mirna_ok & ~disease_ok),
                    ('disease only', ~mirna_ok & disease_ok),
                    ('neither', ~mirna_ok & ~disease_ok)):
    print(f'  {label:28} {int(mask.sum()):6}  {100 * mask.mean():5.1f}%')

associations: 53553
  miRNA resolved                51627   96.4%
  disease resolved              41255   77.0%
  both (usable association)     39915   74.5%
  miRNA only                    11712   21.9%
  disease only                   1340    2.5%
  neither                         586    1.1%


In [22]:
import sqlite3
from textmining.paths import DB_DIR

DB_PATH = DB_DIR / 'atlas_final.db'

# HMDD names a precursor, the db stores precursor and mature as separate
# entities, so record both the precursor accession and its mature children.
mi_to_mimat = defaultdict(set)
with open(mirbase.parent_to_child_path) as f:
    for row in csv.DictReader(f, delimiter='\t'):
        if row['child'].startswith('MIMAT'):
            mi_to_mimat[row['parent']].add(row['child'])

def mirna_accession(raw: str):
    """miRBase accession for an HMDD name, mirroring classify()'s match tiers."""
    name = raw.strip()
    for level in (norm_resources.precursor_normalizer, norm_resources.mature_normalizer):
        if name in level:
            return level[name]['accession']
    folded = fold(name)
    for level in (norm_resources.precursor_normalizer, norm_resources.mature_normalizer):
        if folded in level:
            return level[folded]['accession']
    prefix, _, body = name.partition('-')
    for family in family_folded.get(fold(body), ()):
        accession = norm_resources.family_normalizer[family]
        if prefix in family_prefixes.get(accession, set()):
            return accession
    return None

# same standard as the coverage cell: one identifier per entity, or not counted
df['mir_accession'] = [mirna_accession(n) if ok else None
                       for n, ok in zip(df['miRNA'], mirna_ok)]
df['term_accession'] = [next(iter(ids)) if ok and len(ids) == 1 else None
                        for ids, ok in zip(df['mondo_ids'], disease_ok)]

checkable = df['mir_accession'].notna() & df['term_accession'].notna()
print(f'associations: {len(df)}')
print(f'  resolvable to (miRBase accession, MONDO id): {int(checkable.sum())} '
      f'({100 * checkable.mean():.1f}%)')

associations: 53553
  resolvable to (miRBase accession, MONDO id): 39915 (74.5%)


In [23]:
con = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
entity_id = dict(con.execute('SELECT accession, id FROM entity'))
disease_assoc = {(mir, term): (n_art, n_direct) for mir, term, n_art, n_direct in con.execute(
    'SELECT mirna_entity_id, term_entity_id, n_articles, n_direct_articles '
    "FROM association WHERE term_type = 'DISEASE'")}
print(f'entities: {len(entity_id)}   disease associations in db: {len(disease_assoc)}')

def lookup(mir_accession, term_accession, include_mature):
    """Evidence rows for this pair, optionally counting the precursor's matures."""
    term = entity_id.get(term_accession)
    if term is None:
        return []
    accessions = [mir_accession]
    if include_mature:
        accessions += sorted(mi_to_mimat.get(mir_accession, ()))
    hits = [disease_assoc.get((entity_id[a], term)) for a in accessions if a in entity_id]
    return [h for h in hits if h]

entities: 330687   disease associations in db: 2160540


In [24]:
results = Counter()
for mir, term in zip(df.loc[checkable, 'mir_accession'], df.loc[checkable, 'term_accession']):
    for label, include_mature in (('exact accession', False), ('precursor or matures', True)):
        hits = lookup(mir, term, include_mature)
        if hits:
            results[label] += 1
            if any(direct > 0 for _, direct in hits):
                results[label + ', direct evidence'] += 1

n, total = int(checkable.sum()), len(df)
print(f'of all {total} HMDD associations        (of the {n} resolvable)')
for label in ('exact accession', 'exact accession, direct evidence',
              'precursor or matures', 'precursor or matures, direct evidence'):
    found = results[label]
    print(f'  {label:40} {found:6}  {100 * found / total:5.1f}%   '
          f'({100 * found / n:5.1f}%)')

of all 53553 HMDD associations        (of the 39915 resolvable)
  exact accession                           34598   64.6%   ( 86.7%)
  exact accession, direct evidence          29668   55.4%   ( 74.3%)
  precursor or matures                      36743   68.6%   ( 92.1%)
  precursor or matures, direct evidence     32433   60.6%   ( 81.3%)


In [25]:
# HMDD cites PMIDs, but full texts are stored under their PMC id
pmc = pd.read_csv(res.PMC_METADATA, usecols=['PMCID', 'PMID'], dtype=str)
hmdd_pmids = set(df['PMID'].astype(str).str.strip())
pmid_of = {pmid: pmid for pmid in hmdd_pmids}
pmid_of.update(pmc.loc[pmc['PMID'].isin(hmdd_pmids), ['PMCID', 'PMID']].values)

# PMID -> our article ids (abstract and/or full text)
article_ids = defaultdict(set)
for external_id, aid in con.execute('SELECT external_id, id FROM article'):
    if external_id in pmid_of:
        article_ids[pmid_of[external_id]].add(aid)

mirnas_with_associations = {row[0] for row in con.execute(
    "SELECT DISTINCT mirna_entity_id FROM association WHERE term_type = 'DISEASE'")}
terms_with_associations = {row[0] for row in con.execute(
    "SELECT DISTINCT term_entity_id FROM association WHERE term_type = 'DISEASE'")}
# a miRNA can be mentioned in a paper without forming any association there
mir_mentions = set(con.execute(
    'SELECT mir_key, article_id FROM mir_mention_article WHERE blacklisted = 0'))

resolvable = df[checkable].copy()
resolvable['found'] = [bool(lookup(mir, term, False)) for mir, term
                       in zip(resolvable['mir_accession'], resolvable['term_accession'])]
missed = resolvable[~resolvable['found']]

# the article table holds the papers with at least one detected miRNA
in_db = missed['PMID'].astype(str).str.strip().isin(article_ids)
print(f'resolvable associations: {len(resolvable)}   not found: {len(missed)}')
print(f'  source paper in database:     {int(in_db.sum()):5}  {100 * in_db.mean():5.1f}%')
print(f'  source paper not in database: {int((~in_db).sum()):5}  {100 * (~in_db).mean():5.1f}%')
print()
held = resolvable['PMID'].astype(str).str.strip().isin(article_ids).mean()
print(f'for context, {100 * held:.1f}% of all resolvable HMDD rows cite a paper in the database')

resolvable associations: 39915   not found: 5317
  source paper in database:      5213   98.0%
  source paper not in database:   104    2.0%

for context, 98.4% of all resolvable HMDD rows cite a paper in the database


In [26]:
def miss_reason(mir_accession, term_accession, pmid):
    """Why this HMDD association is absent, from coarsest cause to finest."""
    articles = article_ids.get(str(pmid).strip())
    if not articles:
        return 'paper not in database'
    if entity_id.get(mir_accession) not in mirnas_with_associations:
        return 'miRNA has no disease associations at all'
    if entity_id.get(term_accession) not in terms_with_associations:
        return 'disease term never linked to any miRNA'
    if not any((mir_accession, article) in mir_mentions for article in articles):
        return 'miRNA not detected in that paper'
    return 'both present, pair not extracted'

reasons = Counter(miss_reason(mir, term, pmid) for mir, term, pmid
                  in zip(missed['mir_accession'], missed['term_accession'], missed['PMID']))

print(f'why {len(missed)} resolvable associations are missing:')
for reason, n in reasons.most_common():
    print(f'  {n:5}  {100 * n / len(missed):5.1f}%  {reason}')

why 5317 resolvable associations are missing:
   3014   56.7%  miRNA not detected in that paper
   1353   25.4%  both present, pair not extracted
    792   14.9%  disease term never linked to any miRNA
    104    2.0%  paper not in database
     54    1.0%  miRNA has no disease associations at all


## Relation labels against HMDD codes

HMDD codes that state a direction or a biomarker are mapped to my labels. The other codes (`other`, the target codes, ...) only say that the pair is related and are left out.

Two levels:
- **pair**: the labels of all direct evidence for the miRNA-disease pair;
- **paper**: only the sentences from the paper HMDD cites for that entry.

In [27]:
# HMDD code -> labels counted as agreement; the first one is the expected label
CODE_TO_LABELS = {
    'genetics_overexpression_promote': ('promotes',),
    'genetics_knock down_suppress': ('promotes',),
    'genetics_overexpression_suppress': ('inhibits',),
    'genetics_knock down_promote': ('inhibits',),
    'tissue_expression_up': ('expression_up',),
    'tissue_expression_down': ('expression_down',),
    'circulation_biomarker_diagnosis_ns': ('biomarker',),
    'circulation_biomarker_prognosis_ns': ('biomarker',),
    'circulation_biomarker_diagnosis_up': ('biomarker', 'expression_up'),
    'circulation_biomarker_prognosis_up': ('biomarker', 'expression_up'),
    'circulation_biomarker_diagnosis_down': ('biomarker', 'expression_down'),
    'circulation_biomarker_prognosis_down': ('biomarker', 'expression_down'),
}

df['accepted'] = df['code'].map(CODE_TO_LABELS)
df['hmdd_label'] = df['accepted'].str[0]
labelled = df[checkable & df['accepted'].notna()].copy()
labelled['PMID'] = labelled['PMID'].astype(str).str.strip()

print(f"entries with a mapped code: {int(df['accepted'].notna().sum())}, resolvable: {len(labelled)}")
labelled['hmdd_label'].value_counts()

entries with a mapped code: 14745, resolvable: 11371


hmdd_label
biomarker          5042
promotes           3556
inhibits           2598
expression_up        96
expression_down      79
Name: count, dtype: int64

In [28]:
def accessions(mir_accession):
    """The HMDD miRNA and, for a precursor, its mature forms."""
    return [mir_accession, *sorted(mi_to_mimat.get(mir_accession, ()))]

pairs = {(accession, term)
         for mir, term in zip(labelled['mir_accession'], labelled['term_accession'])
         for accession in accessions(mir)}

con.execute('CREATE TEMP TABLE hmdd_pair(mir TEXT, term TEXT)')
con.executemany('INSERT INTO hmdd_pair VALUES (?, ?)', pairs)
con.execute('CREATE TEMP TABLE hmdd_paper(article_id INTEGER PRIMARY KEY, pmid TEXT)')
con.executemany('INSERT INTO hmdd_paper SELECT id, ? FROM article WHERE external_id = ?',
                      [(pmid, external_id) for external_id, pmid in pmid_of.items()])
con.execute('''CREATE TEMP TABLE hmdd_assoc AS
    SELECT a.id, p.mir, p.term
    FROM hmdd_pair p
    JOIN entity m ON m.accession = p.mir
    JOIN entity t ON t.accession = p.term
    JOIN association a ON a.mirna_entity_id = m.id AND a.term_entity_id = t.id''')

for table in ('hmdd_pair', 'hmdd_assoc', 'hmdd_paper'):
    print(table, con.execute(f'SELECT count(*) FROM {table}').fetchone()[0])

hmdd_pair 20199
hmdd_assoc 14028
hmdd_paper 53605


In [29]:
# label counts: articles for the whole pair, sentences within the cited paper
pair_labels, paper_labels = defaultdict(Counter), defaultdict(Counter)

for mir, term, label, n in con.execute('''
        SELECT h.mir, h.term, al.label, al.n_articles
        FROM hmdd_assoc h JOIN association_label al ON al.association_id = h.id'''):
    pair_labels[mir, term][label] += n

for mir, term, pmid, label, n in con.execute('''
        SELECT h.mir, h.term, p.pmid, mp.label, count(DISTINCT ae.sentence_id)
        FROM hmdd_assoc h
        CROSS JOIN association_evidence ae ON ae.association_id = h.id
        JOIN sentence s ON s.id = ae.sentence_id
        JOIN hmdd_paper p ON p.article_id = s.article_id
        JOIN mention_pair mp ON mp.id = ae.mention_pair_id
        WHERE mp.label IS NOT NULL
        GROUP BY 1, 2, 3, 4'''):
    paper_labels[mir, term, pmid][label] += n

print(f'pairs with labels: {len(pair_labels)}   (pair, paper) with labels: {len(paper_labels)}')

pairs with labels: 11152   (pair, paper) with labels: 36218


In [30]:
def leading(counts):
    """The rule of association.dominant_label: the most frequent specific label,
    'associated' only without one, 'none' only if nothing else."""
    if not counts:
        return None
    return min(counts, key=lambda label: (label == 'none', label == 'associated', -counts[label], label))

def merged(table, mir, *key):
    """Counts summed over the HMDD miRNA and its mature forms."""
    total = Counter()
    for accession in accessions(mir):
        total.update(table.get((accession, *key), {}))
    return total

labelled['pair_counts'] = [merged(pair_labels, mir, term) for mir, term
                           in zip(labelled['mir_accession'], labelled['term_accession'])]
labelled['paper_counts'] = [merged(paper_labels, mir, term, pmid) for mir, term, pmid
                            in zip(labelled['mir_accession'], labelled['term_accession'], labelled['PMID'])]

for level in ('pair', 'paper'):
    counts = labelled[f'{level}_counts']
    labelled[f'{level}_label'] = counts.map(leading)
    labelled[f'{level}_agrees'] = [label in accepted for label, accepted
                                   in zip(labelled[f'{level}_label'], labelled['accepted'])]
    labelled[f'{level}_any'] = [any(c[label] for label in accepted) for c, accepted
                                in zip(counts, labelled['accepted'])]

In [31]:
def agreement(level):
    """Per HMDD label: entries with labelled evidence at this level, and how often my label agrees.

    leading: my leading label is the HMDD one; any: at least one sentence (article) has it."""
    found = labelled[labelled[f'{level}_label'].notna()]
    columns = {'with_evidence': ('code', 'size'),
               'leading_agrees': (f'{level}_agrees', 'mean'),
               'any_agrees': (f'{level}_any', 'mean')}
    table = found.groupby('hmdd_label').agg(**columns)
    table.loc['all'] = found.assign(hmdd_label='all').groupby('hmdd_label').agg(**columns).loc['all']
    table.insert(0, 'hmdd_entries', labelled['hmdd_label'].value_counts())
    table.loc['all', 'hmdd_entries'] = len(labelled)
    return table.astype({'hmdd_entries': int, 'with_evidence': int}).round(3)

agreement('paper')

,hmdd_entries,with_evidence,leading_agrees,any_agrees
hmdd_label,,,,
biomarker,5042,2454,0.559,0.652
expression_down,79,32,0.531,0.750
expression_up,96,31,0.677,0.774
inhibits,2598,1142,0.495,0.693
promotes,3556,1507,0.203,0.485
all,11371,5166,0.441,0.614


In [32]:
agreement('pair')

,hmdd_entries,with_evidence,leading_agrees,any_agrees
hmdd_label,,,,
biomarker,5042,4353,0.399,0.830
expression_down,79,69,0.275,0.681
expression_up,96,60,0.383,0.767
inhibits,2598,2088,0.518,0.818
promotes,3556,2876,0.135,0.709
all,11371,9446,0.344,0.789


In [33]:
# rows: HMDD label, columns: my leading label in the cited paper
in_paper = labelled[labelled['paper_label'].notna()]
pd.crosstab(in_paper['hmdd_label'], in_paper['paper_label'], margins=True)

paper_label,associated,biomarker,expression_down,expression_up,inhibits,none,promotes,All
hmdd_label,,,,,,,,
biomarker,173,1101,308,431,166,161,114,2454
expression_down,3,2,17,2,6,0,2,32
expression_up,1,0,2,21,2,2,3,31
inhibits,68,86,104,52,565,24,243,1142
promotes,81,77,125,89,807,22,306,1507
All,326,1266,556,595,1546,209,668,5166


In [34]:
# the same as shares of each HMDD label
pd.crosstab(in_paper['hmdd_label'], in_paper['paper_label'], normalize='index').round(2)

paper_label,associated,biomarker,expression_down,expression_up,inhibits,none,promotes
hmdd_label,,,,,,,
biomarker,0.07,0.45,0.13,0.18,0.07,0.07,0.05
expression_down,0.09,0.06,0.53,0.06,0.19,0.00,0.06
expression_up,0.03,0.00,0.06,0.68,0.06,0.06,0.10
inhibits,0.06,0.08,0.09,0.05,0.49,0.02,0.21
promotes,0.05,0.05,0.08,0.06,0.54,0.01,0.20


### Do the genetics codes state the miRNA's direction?

The four `genetics_*` codes are split out below, with a few HMDD sentences per code to read.
If all four rows look alike, the codes do not encode whether the miRNA promotes or inhibits the disease,
and the `promotes` / `inhibits` rows above say nothing about my labels.

In [35]:
genetics = in_paper[in_paper['code'].str.startswith('genetics')]
pd.crosstab(genetics['code'], genetics['paper_label'], normalize='index').round(2)

paper_label,associated,biomarker,expression_down,expression_up,inhibits,none,promotes
code,,,,,,,
genetics_knock down_promote,0.07,0.06,0.08,0.05,0.45,0.02,0.27
genetics_knock down_suppress,0.08,0.02,0.11,0.03,0.44,0.02,0.30
genetics_overexpression_promote,0.05,0.06,0.08,0.06,0.56,0.01,0.18
genetics_overexpression_suppress,0.03,0.12,0.11,0.04,0.60,0.01,0.08


In [36]:
for code, rows in labelled[labelled['code'].str.startswith('genetics')].groupby('code'):
    print(f'\n{code}')
    for row in rows.sample(8, random_state=0).itertuples():
        print(f'  {row.miRNA} | {row.disease} | {row.description[:250]}')


genetics_knock down_promote
  hsa-mir-520a | Carcinoma, Non-Small-Cell Lung | Additionally, rescue assay exhibited the anticancer activity mediated by SNHG16 knockdown on NSCLC could be reversed by miR-520a-3p inhibition or EphA2 overexpression.
  hsa-mir-204 | Glioma | In conclusion, knockdown of HOXD-AS1 suppressed proliferation, migration and invasion and enhanced DDP sensitivity of glioma cells through sequestering miR-204, providing a promising therapeutic target for glioma patients.
  hsa-mir-154 | Carcinoma, Hepatocellular | CONCLUSION: DLGAP1-AS2 knockdown may inhibit HCC cell migration and invasion by regulating miR
  hsa-mir-149 | Colorectal Neoplasms | These new findings indicate that DLGAP1-AS1 knockdown inhibited the progression of CRC and enhanced the 5-FU sensitivity of CRC cells through miR
  hsa-mir-421 | Adenocarcinoma of Lung | Gain- and loss-of-function assays showed that miR-421 promoted LUAD cell migration and invasion.
  hsa-mir-433 | Carcinoma, Non-Small-Cell L

## Label present in the cited paper

Share of entries whose cited paper has at least one sentence with the label, for entries whose code states the label vs. all other coded entries.

In [37]:
with_evidence = labelled[labelled['paper_counts'].map(bool)]
rows = {}
for label in ('biomarker', 'expression_up', 'expression_down'):
    states = with_evidence['accepted'].map(lambda accepted: label in accepted)
    present = with_evidence['paper_counts'].map(lambda counts: counts[label] > 0)
    rows[label] = {'entries': int(states.sum()), 'present': present[states].mean(),
                   'other_entries': int((~states).sum()), 'present_other': present[~states].mean()}
pd.DataFrame(rows).T.astype({'entries': int, 'other_entries': int}).round(3)

,entries,present,other_entries,present_other
biomarker,2454,0.557,2712,0.236
expression_up,996,0.522,4170,0.278
expression_down,187,0.727,4979,0.360
